# Workshop 04: Evaluating Forecasts & Walk-Forward Validation
**Machine Learning II: Forecasting Economic Time Series — Lazarski University**

### Learning Objectives:
1. **Implement Core Point Accuracy Metrics:** Vectorize MAE, RMSE, MAPE (with epsilon zero-guard), and MASE from scratch in pure Python.
2. **Quantify Uncertainty:** Implement Pinball Loss for conditional quantiles and the Winkler Score for prediction intervals.
3. **Build Walk-Forward Validation Engines:** Construct expanding and rolling origin split generators that strictly prevent future information leakage.
4. **Prove the Random K-Fold Crime:** Empirically demonstrate why shuffled K-fold cross-validation deflates error and produces fraudulent $R^2$ on autocorrelated time series.
5. **Conduct Hypothesis Testing:** Implement the Diebold–Mariano (1995) test with Newey-West HAC long-run variance and the Harvey-Leybourne-Newbold (1997) finite-sample correction.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit
from pathlib import Path
from typing import List, Tuple, Dict, Any, Callable, Optional

# Set consistent plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
print('Workshop 04 environment initialized.')


---
## Milestone 1: Robust Macroeconomic Data Ingestion
Implement `load_macro_series` to ingest macroeconomic time series from CSV, enforce a monotonic `pd.DatetimeIndex`, and assign the specified calendar frequency (`MS` for monthly, `QS` for quarterly).


In [ ]:
def load_macro_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: Optional[str] = None
) -> pd.Series:
    """
    Ingest a macroeconomic CSV file and enforce a clean DatetimeIndex and target frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Ingest macroeconomic benchmark series
rsafs = load_macro_series(str(DATA_DIR / 'RSAFS.csv'), 'observation_date', 'RSAFSNA', freq='MS')
indpro = load_macro_series(str(DATA_DIR / 'INDPRO.csv'), 'observation_date', 'INDPRO', freq='MS')
beer = load_macro_series(str(DATA_DIR / 'aus_beer.csv'), 'observation_date', 'beer_production', freq='QS')
intermittent = load_macro_series(str(DATA_DIR / 'intermittent_sales.csv'), 'observation_date', 'sales', freq='MS')

print(f'RSAFS loaded: {len(rsafs)} monthly obs from {rsafs.index.min().strftime("%Y-%m")} to {rsafs.index.max().strftime("%Y-%m")}')
print(f'INDPRO loaded: {len(indpro)} monthly obs')
print(f'Australian Beer loaded: {len(beer)} quarterly obs')
print(f'Intermittent Sales loaded: {len(intermittent)} obs (contains near-zero points: {intermittent.values[2]})')


---
## Milestone 2: Vectorized Point Accuracy Metrics (MAE, RMSE, MAPE)
Implement:
1. `compute_mae(y_true, y_pred)`: $\text{MAE} = \frac{1}{H}\sum |y_t - \hat{y}_t|$ (minimizes conditional median).
2. `compute_rmse(y_true, y_pred)`: $\text{RMSE} = \sqrt{\frac{1}{H}\sum (y_t - \hat{y}_t)^2}$ (minimizes conditional mean).
3. `compute_mape(y_true, y_pred, eps=1e-5)`: $\text{MAPE} = \frac{100}{H}\sum \left|\frac{y_t - \hat{y}_t}{\max(|y_t|, \varepsilon)}\right|$ with epsilon protection against zero-division.


In [ ]:
def compute_mae(y_true, y_pred) -> float:
    """
    Compute Mean Absolute Error (MAE) under L1 loss.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")

def compute_rmse(y_true, y_pred) -> float:
    """
    Compute Root Mean Squared Error (RMSE) under L2 loss.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")

def compute_mape(y_true, y_pred, eps: float = 1e-5) -> float:
    """
    Compute Mean Absolute Percentage Error (MAPE) with epsilon protection against zero-division.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: The Near-Zero Division Trap
# We forecast intermittent sales using a simple historical mean
train_inter = intermittent.iloc[:24]
test_inter = intermittent.iloc[24:]
pred_mean = np.full(len(test_inter), train_inter.mean())

mae_val = compute_mae(test_inter, pred_mean)
rmse_val = compute_rmse(test_inter, pred_mean)
mape_val = compute_mape(test_inter, pred_mean)

print('--- Intermittent Sales Error Metrics ---')
print(f'Mean Absolute Error (MAE):       {mae_val:.2f} units')
print(f'Root Mean Squared Error (RMSE):  {rmse_val:.2f} units')
print(f'Mean Absolute Pct Error (MAPE):  {mape_val:.2f}% (EXPLOSION due to near-zero point!)')


---
## Milestone 3: Mean Absolute Scaled Error (MASE)
Implement `compute_mase(y_train, y_test, y_pred, m=1)`.
Rob Hyndman's MASE normalizes test absolute errors by the in-sample Mean Absolute Error of a one-step Naive (or Seasonal Naive with period $m$) benchmark:
$$Q = \frac{1}{T-m}\sum_{j=m+1}^T |y_j - y_{j-m}|, \quad \text{MASE} = \frac{\text{mean}(|y_{\text{test}} - \hat{y}|)}{Q}$$
- **Hurdle:** $\text{MASE} < 1.0$ beats Naive; $\text{MASE} > 1.0$ is worse than Naive.
- **Scale-Free:** Works across series of completely different units.


In [ ]:
def compute_mase(y_train, y_test, y_pred, m: int = 1) -> float:
    """
    Compute Mean Absolute Scaled Error (MASE) using in-sample Naive or Seasonal Naive denominator.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: MASE on Australian Beer (m=4) and Intermittent Sales (m=1)
beer_train, beer_test = beer.iloc[:-8], beer.iloc[-8:]
beer_snaive_pred = beer_train.iloc[-4:].tolist() * 2  # Repeat last 4 quarters twice

mase_beer = compute_mase(beer_train, beer_test, beer_snaive_pred, m=4)
mase_inter = compute_mase(train_inter, test_inter, pred_mean, m=1)

print(f'Beer Seasonal Naive MASE:        {mase_beer:.3f} (Benchmark baseline)')
print(f'Intermittent Mean Forecast MASE: {mase_inter:.3f} (Stable, zero-safe, no explosion!)')


---
## Milestone 4: Distributional & Interval Scoring
Implement:
1. `compute_pinball_loss(y_true, q_pred, alpha)`: Asymmetric quantile loss targeting conditional percentile $\alpha \in (0, 1)$:
   $$L_\alpha(u) = \max(\alpha u, (\alpha - 1) u), \quad u = y - \hat{q}_\alpha$$
2. `compute_winkler_score(y_true, lower, upper, alpha=0.05)`: Interval score balancing bandwidth against under-coverage penalties:
   $$W_\alpha(L, U, y) = (U - L) + \frac{2}{\alpha}(L - y)\mathbf{1}(y < L) + \frac{2}{\alpha}(y - U)\mathbf{1}(y > U)$$


In [ ]:
def compute_pinball_loss(y_true, q_pred, alpha: float) -> float:
    """
    Compute asymmetric quantile (pinball) loss for target quantile alpha in (0, 1).
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")

def compute_winkler_score(y_true, lower, upper, alpha: float = 0.05) -> float:
    """
    Compute Winkler interval score balancing bandwidth against under-coverage penalties.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Scoring Prediction Intervals on Retail Sales
train_r = rsafs.iloc[:324]
test_r = rsafs.iloc[324:348]

step_std = np.std(np.diff(train_r), ddof=1)
last_val = train_r.iloc[-1]
naive_point = np.full(len(test_r), last_val)
lower_95 = naive_point - 1.96 * step_std * np.sqrt(np.arange(1, len(test_r) + 1))
upper_95 = naive_point + 1.96 * step_std * np.sqrt(np.arange(1, len(test_r) + 1))

winkler_95 = compute_winkler_score(test_r, lower_95, upper_95, alpha=0.05)
q90_loss = compute_pinball_loss(test_r, upper_95, alpha=0.975)

print(f'95% Prediction Interval Winkler Score: {winkler_95:.2f}')
print(f'97.5th Percentile Pinball Loss:        {q90_loss:.2f}')


---
## Milestone 5: Time Series Cross-Validation (Walk-Forward Engine)
Implement `generate_walk_forward_splits(n_samples, min_train, forecast_horizon, step=1, mode="expanding")`.
Generates sequential `(train_indices, test_indices)` splits advancing chronologically without future leakage.


In [ ]:
def generate_walk_forward_splits(
    n_samples: int,
    min_train: int,
    forecast_horizon: int,
    step: int = 1,
    mode: str = "expanding"
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Generate sequential train/test indices for walk-forward time series cross-validation.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Inspect expanding vs rolling walk-forward origins
splits_exp = generate_walk_forward_splits(n_samples=50, min_train=30, forecast_horizon=6, step=4, mode='expanding')
splits_rol = generate_walk_forward_splits(n_samples=50, min_train=30, forecast_horizon=6, step=4, mode='rolling')

print(f'Generated {len(splits_exp)} expanding folds:')
for idx, (tr, te) in enumerate(splits_exp):
    print(f'  Fold {idx+1}: Train [{tr[0]}..{tr[-1]}] (size {len(tr)}) | Test [{te[0]}..{te[-1]}] (size {len(te)})')

print(f'\nGenerated {len(splits_rol)} rolling folds (fixed window = 30):')
for idx, (tr, te) in enumerate(splits_rol):
    print(f'  Fold {idx+1}: Train [{tr[0]}..{tr[-1]}] (size {len(tr)}) | Test [{te[0]}..{te[-1]}] (size {len(te)})')


---
## Milestone 6: Proving the Random K-Fold Leakage Crime
Implement `demonstrate_kfold_leakage(series, n_splits=5, horizon=12)`.
Compares shuffled 1-step K-fold against honest multi-step recursive walk-forward validation on a persistent AR(1) process ($y_t = 0.95 y_{t-1} + \varepsilon_t$).


In [ ]:
def demonstrate_kfold_leakage(
    series: pd.Series,
    n_splits: int = 5,
    horizon: int = 12
) -> Dict[str, float]:
    """
    Demonstrate that random K-fold shuffles time and produces fraudulent R^2 on persistent series.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Run K-Fold Leakage Experiment on Synthetic AR(1) Series
np.random.seed(42)
T = 120
eps = np.random.normal(0, 1.0, T)
y_ar1 = np.zeros(T)
for t in range(1, T):
    y_ar1[t] = 0.95 * y_ar1[t - 1] + eps[t]

leakage_audit = demonstrate_kfold_leakage(pd.Series(y_ar1), n_splits=5, horizon=12)
print('--- Autocorrelation Leakage Forensics ---')
print(f'Shuffled K-Fold Test R^2:        {leakage_audit["kfold_r2"]:.4f}  (FRAUDULENT: Interpolating future points)')
print(f'Walk-Forward Recursive Test R^2: {leakage_audit["walk_forward_r2"]:.4f}  (HONEST: True out-of-sample difficulty)')
print(f'Leakage Inflation Gap:           {leakage_audit["leakage_gap"]:.4f}  (Artificially deflated error)')


---
## Milestone 7: Horizon Degradation Profiles
Implement `evaluate_horizon_degradation(y, model_fn, min_train, max_h)`.
Computes MAE and RMSE across lead times $h = 1 \dots H$ to reveal how predictive uncertainty compounds over time.


In [ ]:
def evaluate_horizon_degradation(
    y: pd.Series,
    model_fn: Callable,
    min_train: int,
    max_h: int
) -> pd.DataFrame:
    """
    Compute forecast error degradation profiles across lead times h = 1 .. max_h.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Compute Horizon Degradation for Naive Forecast on INDPRO
def naive_benchmark_fn(train, steps):
    return np.full(steps, train[-1])

deg_df = evaluate_horizon_degradation(indpro.iloc[-120:], naive_benchmark_fn, min_train=60, max_h=24)
print('INDPRO Horizon Error Degradation Profile (h = 1..24):')
print(deg_df.head(6))
print(deg_df.tail(6))


---
## Milestone 8: Statistical Hypothesis Testing (Diebold–Mariano Test)
Implement `diebold_mariano_test(y_true, y_pred1, y_pred2, h=1, loss="squared", small_sample=True)`.
Tests the formal null hypothesis $H_0: \mathbb{E}[d_t] = 0$ on the loss differential series $d_t = L(e_{1,t}) - L(e_{2,t})$.
- Estimates long-run spectral variance using Newey-West Bartlett kernel weighting up to lag $h-1$.
- Applies the Harvey-Leybourne-Newbold (1997) finite-sample correction multiplier:
  $$\text{HLN} = DM \cdot \sqrt{\frac{P + 1 - 2h + h(h-1)/P}{P}}$$
- Compares against Student-$t$ distribution with $P-1$ degrees of freedom.


In [ ]:
def diebold_mariano_test(
    y_true,
    y_pred1,
    y_pred2,
    h: int = 1,
    loss: str = "squared",
    small_sample: bool = True
) -> Tuple[float, float, bool]:
    """
    Execute Diebold-Mariano test with Newey-West HAC variance and HLN finite-sample correction.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Diebold-Mariano Test on Retail Sales Holdout
train_full = rsafs.iloc[:324]
test_holdout = rsafs.iloc[324:384]  # P = 60 months

# Model 1: Drift Benchmark
drift_slope = (train_full.iloc[-1] - train_full.iloc[0]) / (len(train_full) - 1)
pred_drift = train_full.iloc[-1] + drift_slope * np.arange(1, len(test_holdout) + 1)

# Model 2: Mean Benchmark
pred_mean_r = np.full(len(test_holdout), train_full.mean())

dm_stat, dm_pval, dm_sig = diebold_mariano_test(test_holdout, pred_drift, pred_mean_r, h=4, small_sample=True)

rmse_m1 = compute_rmse(test_holdout, pred_drift)
rmse_m2 = compute_rmse(test_holdout, pred_mean_r)

print('--- Diebold-Mariano Predictive Superiority Test ---')
print(f'Model 1 (Drift) RMSE: {rmse_m1:.2f} | Model 2 (Mean) RMSE: {rmse_m2:.2f}')
print(f'HLN-Adjusted DM Statistic: {dm_stat:.3f}')
print(f'Two-Sided p-value:         {dm_pval:.4e}')
print(f'Statistically Significant: {dm_sig} (Model 1 is decisively superior!)')


---
## Milestone 9: Production Leakage Audit Suite
Implement `run_leakage_audit_suite(train_features, test_features, model)` to verify temporal boundaries, partition disjointness, and column alignment.


In [ ]:
def run_leakage_audit_suite(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    model: Any = None
) -> Dict[str, bool]:
    """
    Audit training and test splits for temporal peeking, scaler leakage, and future imputation.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Audit Retail Sales Feature Splits
df_feat = pd.DataFrame({'lag1': rsafs.shift(1), 'lag12': rsafs.shift(12)}).dropna()
tr_feat = df_feat.iloc[:300]
te_feat = df_feat.iloc[300:]

audit_report = run_leakage_audit_suite(tr_feat, te_feat)
print('Leakage Audit Report:')
for k, v in audit_report.items():
    print(f'  {k}: {"PASS (True)" if v else "FAIL (False)"}')
